# LangChain 101

Using **Google Gemini** as the model (free API key, no billing required).

1. Go to https://aistudio.google.com/apikey
2. Click "Create API key"
3. Run the next cell and paste the key when prompted (it won't be echoed, and we save it to a local `.env` file so you only have to do this once)


In [17]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # loads variables from a .env file in this directory, if present

if not os.environ.get("GOOGLE_API_KEY"):
    key = getpass("Enter your Google AI Studio API key: ")
    os.environ["GOOGLE_API_KEY"] = key
    # persist it so future notebook runs don't ask again
    with open(".env", "a") as f:
        f.write(f"\nGOOGLE_API_KEY={key}\n")

print("API key is set:", bool(os.environ.get("GOOGLE_API_KEY")))


API key is set: True


## 1. Your first model call

`init_chat_model` gives you a provider-agnostic way to get a chat model. `invoke()` sends a single request and blocks until the full response comes back.

In [10]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

response = llm.invoke("What is LangChain, in one sentence?")
print(response.content)


[{'type': 'text', 'text': 'LangChain is a framework that simplifies the development of applications powered by large language models (LLMs) by enabling them to connect with external data sources, APIs, and memory.', 'extras': {'signature': 'EmAKXgFpFH0TIZjoJ0Qd0Nav3TAF7FZv2GkUe4VcrZI7xFmkdvS8/4ZgdXvN+fsHfu6lkw27WC9YHj9s95fIwNIRg7og5vUzmr20W665kC6YI21J/yPffMTmExSYZMr2V3Y='}}]


## 2. Messages

Chat models take a list of messages, not just a string. `SystemMessage` sets behavior/persona, `HumanMessage` is user input, `AIMessage` is the model's reply. You build up conversation history by appending to this list.

In [11]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a terse pirate. Answer in one short sentence, in pirate speak."),
    HumanMessage("What is the capital of France?"),
]

response = llm.invoke(messages)
print(response.content)

# continue the conversation by appending the reply and a new question
messages.append(AIMessage(response.content))
messages.append(HumanMessage("What about Japan?"))

response = llm.invoke(messages)
print(response.content)


[{'type': 'text', 'text': 'Paris, ye scallywag!', 'extras': {'signature': 'EmAKXgFpFH0TIJay0tBpUBfywfTjhRbngklTOIgn7rXBZCozR+Z5T2+jRaZTpwlv0so/Txzepa/ZDCxbxRdsRNuPf3z4ySkEbkK+H5gzz/5LPdZSTbktw1SB9cZOIMcYWwQ='}}]
[{'type': 'text', 'text': "Tokyo be the capital o' Japan, savvy?", 'extras': {'signature': 'EmAKXgFpFH0TzQ4u1tfM1Z/K3VILpelDpIAKX0EN3vXoe8uzGFckBd/mh+TVfrdFjc+m41JLPHlU+dI0f7ZK56bR+Vf7OAU79drQVP3LcrfCmqtd9z6081qcNPCzL3an0/k='}}]


## 3. Prompt templates

`ChatPromptTemplate` lets you write a reusable prompt with `{placeholders}` filled in at call time, instead of hand-building message lists.

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains things to a {audience}."),
    ("human", "Explain {topic} in 2-3 sentences."),
])

filled = prompt.invoke({"audience": "10 year old", "topic": "how the internet works"})
print(filled)  # see the actual list of messages this produces

response = llm.invoke(filled)
print("\n---\n")
print(response.content)


messages=[SystemMessage(content='You are a helpful assistant that explains things to a 10 year old.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain how the internet works in 2-3 sentences.', additional_kwargs={}, response_metadata={})]

---

[{'type': 'text', 'text': 'Imagine the internet as a giant invisible postal service that connects millions of computers all over the world using cables and Wi-Fi. When you want to see a video or a game, your computer sends tiny pieces of information called "packets" zooming through these wires to a giant computer called a server, which sends the game right back to you in a split second!', 'extras': {'signature': 'EmAKXgFpFH0TjCs5ddfxnkSLq/HkIGwd9lu/OvDrjrEgxWaHCY76pvgsSVjGQrpd/ToxPSe/4R5YyzP/b7K9wrkgbA7ysSSnUMbbn4nCzL6Cb/Mb573OQlB9px5vKRzwUO4='}}]


## 4. Chains (LCEL: the `|` pipe operator)

LangChain lets you compose a prompt, a model, and an output parser into a single pipeline using `|`. This is the "LangChain Expression Language" (LCEL) and is the core pattern you'll see everywhere, including as the mental model for LangGraph nodes later on.

`StrOutputParser` just extracts `.content` (the plain string) from the model's response, so `chain.invoke(...)` returns a string instead of a message object.

In [13]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"audience": "senior software engineer", "topic": "how the internet works"})
print(result)


The internet is a global network of interconnected computers that communicate by exchanging packets of data using standardized protocols like TCP/IP. Hardware routers and switches direct this traffic across physical infrastructure—ranging from copper wires to undersea fiber optics—routing data dynamically from source to destination. Applications then interpret these payloads, enabling services like the World Wide Web, DNS resolution, and peer-to-peer communication.


## 5. Streaming

Instead of waiting for the whole response, `.stream()` yields chunks as they're generated — this is what powers "typewriter" UIs. Any chain (not just a raw model) supports `.stream()`.

In [15]:
for chunk in chain.stream({"audience": "curious 8 year old", "topic": "black holes"}):
    print(chunk, end="", flush=True)


Imagine a giant monster in space with a pull so strong that even light can't escape it! Because not even light can bounce off it, we can't see them directly, but they act like cosmic vacuum cleaners. They are made when giant, heavy stars crash and squeeze themselves into a tiny, super-powerful dot.

## 6. Structured output

Instead of parsing free-form text, you can have the model return data matching a schema you define with a Pydantic model. This is essential once you start building agents (LangGraph) that need to make decisions based on model output.

In [16]:
from pydantic import BaseModel, Field

class MovieRecommendation(BaseModel):
    title: str = Field(description="The movie's title")
    year: int = Field(description="Release year")
    reason: str = Field(description="One sentence on why it fits the request")

structured_llm = llm.with_structured_output(MovieRecommendation)

result = structured_llm.invoke("Recommend a sci-fi movie for someone who loved The Matrix.")
print(result)
print(type(result))
print(result.title, "-", result.year)


title='Dark City' year=1998 reason='It shares a remarkably similar mind-bending, reality-questioning neo-noir atmosphere and visual aesthetic just before The Matrix popularized the trope.'
<class '__main__.MovieRecommendation'>
Dark City - 1998


## Next up

You now have the LangChain building blocks: models, messages, prompt templates, LCEL chains, streaming, and structured output. Natural next steps in a follow-up notebook:

- **Tools** — give the model functions it can call (`@tool`, `.bind_tools()`)
- **Tool-calling agent loop** — model decides which tool to call and when
- **LangGraph** — build that agent loop explicitly as a graph of nodes/edges, with state, branching, and persistence — this is where LangChain and LangGraph connect